In [0]:
# ============================================================
# Notebook: 02_silver_transactions_incremental_merge
# Purpose : Load PaySim transactions incrementally from Bronze
#           to Silver using watermark + transaction_id + MERGE.
# ============================================================

from pyspark.sql.functions import col, current_timestamp, sha2, concat_ws
from delta.tables import DeltaTable

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

BRONZE_TABLE = "data_engineering.bronze_layer.finshield_bronze"
SILVER_TABLE = "data_engineering.silver_layer.finshield_silver"
CONTROL_TABLE = "data_engineering.table_metadata.finshield_control"

PIPELINE_NAME = "silver_transactions_incremental"
SOURCE_NAME = "paysim_transactions"

BATCH_SIZE = 50

# ------------------------------------------------------------
# 2. Read last processed watermark
# ------------------------------------------------------------

control_df = spark.sql(f"""
SELECT last_watermark
FROM {CONTROL_TABLE}
WHERE pipeline_name = '{PIPELINE_NAME}'
AND source_name = '{SOURCE_NAME}'
""")

last_watermark = control_df.collect()[0]["last_watermark"]
next_watermark = last_watermark + BATCH_SIZE

print(f"Last watermark: {last_watermark}")
print(f"Processing steps > {last_watermark} and <= {next_watermark}")

# ------------------------------------------------------------
# 3. Read current incremental batch from Bronze
# ------------------------------------------------------------

bronze_df = spark.table(BRONZE_TABLE)

batch_df = bronze_df.filter(
    (col("step") > last_watermark) &
    (col("step") <= next_watermark)
)

batch_count = batch_df.count()
print(f"Batch record count: {batch_count}")

# ------------------------------------------------------------
# 4. Stop safely if no records exist for current batch
# ------------------------------------------------------------

if batch_count == 0:
    print("No new records found. Load skipped.")

else:
    # --------------------------------------------------------
    # 5. Clean and standardize data for Silver layer
    # --------------------------------------------------------

    silver_clean_df = batch_df.select(
        col("step").cast("bigint").alias("step"),
        col("type").alias("transaction_type"),
        col("amount").cast("double").alias("amount"),

        col("nameOrig").alias("origin_account_id"),
        col("oldbalanceOrg").cast("double").alias("origin_old_balance"),
        col("newbalanceOrig").cast("double").alias("origin_new_balance"),

        col("nameDest").alias("destination_account_id"),
        col("oldbalanceDest").cast("double").alias("destination_old_balance"),
        col("newbalanceDest").cast("double").alias("destination_new_balance"),

        col("isFraud").cast("int").alias("is_fraud"),
        col("isFlaggedFraud").cast("int").alias("is_flagged_fraud")
    )

    # --------------------------------------------------------
    # 6. Create transaction_id
    #    PaySim has no natural transaction ID.
    #    So we create a deterministic hash using transaction
    #    attributes. Same transaction = same transaction_id.
    # --------------------------------------------------------

    silver_clean_df = silver_clean_df.withColumn(
        "transaction_id",
        sha2(
            concat_ws(
                "||",
                col("step").cast("string"),
                col("transaction_type"),
                col("amount").cast("string"),
                col("origin_account_id"),
                col("destination_account_id"),
                col("origin_old_balance").cast("string"),
                col("origin_new_balance").cast("string"),
                col("destination_old_balance").cast("string"),
                col("destination_new_balance").cast("string"),
                col("is_fraud").cast("string"),
                col("is_flagged_fraud").cast("string")
            ),
            256
        )
    ).withColumn(
        "silver_load_ts",
        current_timestamp()
    )

    # --------------------------------------------------------
    # 7. First run: create Silver table if it does not exist
    # --------------------------------------------------------

    if not spark.catalog.tableExists(SILVER_TABLE):
        silver_clean_df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(SILVER_TABLE)

        print("Silver table created and first batch loaded.")

    else:
        # ----------------------------------------------------
        # 8. Subsequent runs: MERGE into existing Silver table
        #    Match condition: transaction_id
        #    Existing transaction_id → update
        #    New transaction_id      → insert
        # ----------------------------------------------------

        target_table = DeltaTable.forName(spark, SILVER_TABLE)

        target_table.alias("target").merge(
            silver_clean_df.alias("source"),
            "target.transaction_id = source.transaction_id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

        print("MERGE completed successfully.")

    # --------------------------------------------------------
    # 9. Update watermark only after successful write/MERGE
    # --------------------------------------------------------

    spark.sql(f"""
    UPDATE {CONTROL_TABLE}
    SET last_watermark = {next_watermark},
        updated_at = current_timestamp()
    WHERE pipeline_name = '{PIPELINE_NAME}'
    AND source_name = '{SOURCE_NAME}'
    """)

    print(f"Watermark updated to {next_watermark}")

# ------------------------------------------------------------
# 10. Validation
# ------------------------------------------------------------

print("Control table status:")
spark.sql(f"""
SELECT *
FROM {CONTROL_TABLE}
WHERE pipeline_name = '{PIPELINE_NAME}'
AND source_name = '{SOURCE_NAME}'
""").display()

print("Silver table summary:")
spark.sql(f"""
SELECT
    MIN(step) AS min_step,
    MAX(step) AS max_step,
    COUNT(*) AS total_records,
    COUNT(DISTINCT transaction_id) AS distinct_transactions
FROM {SILVER_TABLE}
""").display()

SQL code for Merge

In [0]:
#silver_clean_df.createOrReplaceTempView("silver_temp_view")
#%sql
#MERGE INTO data_engineering.silver_layer.finshield_silver AS target
#USING silver_temp_view AS source
#ON target.transaction_id = source.transaction_id
#WHEN MATCHED THEN UPDATE SET *
#WHEN NOT MATCHED THEN INSERT *